# Identification of a Superbright Nova in Our Galaxy: Revisiting the Light-curve Analyses of the Candidate Novae CP Lac, CP Pup, V838 Her, V597 Pup, V5583 Sgr, and V5589 Sgr

Paper: https://iopscience.iop.org/article/10.3847/1538-4357/ae96a8

# Part 1. How a Nova Becomes Superbright

In [ ]:
# ============================================================
# WHY CAN A NOVA BECOME SUPERBRIGHT?
# ============================================================
#
# Single Jupyter cell
#
# Output:
#   superbright_nova.webm
#   or
#   superbright_nova.mp4
#
# Requires:
#   numpy
#   matplotlib
#   ffmpeg
#
# Scientific concept:
#   ACCRETION
#       ->
#   THERMONUCLEAR RUNAWAY
#       ->
#   NOVA WIND
#       ->
#   SHOCK FORMATION
#       ->
#   OPTICALLY THICK SHOCKED SHELL
#       ->
#   SUPERBRIGHT NOVA
#       ->
#   SHELL EXPANSION / OPTICALLY THIN
#
# IMPORTANT:
# The geometry and particle distribution are schematic.
# They are intended to illustrate the physical mechanism,
# not to reproduce a hydrodynamic simulation.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, FancyBboxPatch, Arc
from IPython.display import Video, display
import matplotlib.patheffects as pe
import shutil
import os


# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "webm"          # "webm" or "mp4"

FPS = 30
DURATION = 18            # seconds
DPI = 120

NFRAMES = FPS * DURATION

FIG_W = 12.8
FIG_H = 7.2


if OUTPUT.lower() == "webm":
    OUTFILE = "superbright_nova.webm"
else:
    OUTFILE = "superbright_nova.mp4"


if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg not found. Install ffmpeg and make sure "
        "it is available in PATH."
    )


rng = np.random.default_rng(42)


# ============================================================
# HELPERS
# ============================================================

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def pulse(frame, speed=0.1):
    return 0.5 + 0.5 * np.sin(frame * speed)


def hex_to_rgb(h):
    h = h.lstrip("#")
    return np.array(
        [int(h[i:i+2], 16) / 255 for i in (0, 2, 4)]
    )


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="#02050b"
)

ax = fig.add_axes([
    0.03,
    0.12,
    0.94,
    0.78
])

ax.set_facecolor("#02050b")
ax.set_xlim(-1.65, 1.65)
ax.set_ylim(-0.95, 0.95)
ax.set_aspect("equal")
ax.axis("off")


# ============================================================
# BACKGROUND STARS
# ============================================================

N_BG = 230

bg_x = rng.uniform(-1.65, 1.65, N_BG)
bg_y = rng.uniform(-0.95, 0.95, N_BG)

bg_s = rng.uniform(0.3, 2.5, N_BG)

bg_colors = np.ones((N_BG, 4))
bg_colors[:, 3] = rng.uniform(0.04, 0.20, N_BG)

ax.scatter(
    bg_x,
    bg_y,
    s=bg_s,
    c=bg_colors,
    linewidths=0,
    zorder=0
)


# ============================================================
# TITLE
# ============================================================

fig.text(
    0.055,
    0.945,
    "HOW A NOVA BECOMES SUPERBRIGHT",
    color="white",
    fontsize=17,
    weight="bold"
)

fig.text(
    0.055,
    0.910,
    "Formation of an optically thick shocked shell",
    color="#8195a8",
    fontsize=9
)


# ============================================================
# CENTRAL WHITE DWARF
# ============================================================

WD_X = 0.0
WD_Y = 0.02


# Large diffuse glows
wd_glows = []

for size, alpha in [
    (1500, 0.015),
    (900, 0.025),
    (500, 0.045),
    (260, 0.08),
]:

    g = ax.scatter(
        [WD_X],
        [WD_Y],
        s=size,
        c="white",
        alpha=alpha,
        linewidths=0,
        zorder=5
    )

    wd_glows.append(g)


wd_star = ax.scatter(
    [WD_X],
    [WD_Y],
    s=65,
    c="white",
    edgecolors="#bfeaff",
    linewidths=0.8,
    zorder=10
)


# ============================================================
# COMPANION STAR
# ============================================================

COMP_X = -0.82
COMP_Y = 0.08


comp_glows = []

for size, alpha in [
    (950, 0.018),
    (500, 0.035),
    (250, 0.07),
]:

    g = ax.scatter(
        [COMP_X],
        [COMP_Y],
        s=size,
        c="#ff9a56",
        alpha=alpha,
        linewidths=0,
        zorder=3
    )

    comp_glows.append(g)


comp_star = ax.scatter(
    [COMP_X],
    [COMP_Y],
    s=130,
    c="#ffad68",
    edgecolors="#ffd0a8",
    linewidths=0.8,
    zorder=7
)


# ============================================================
# ACCRETION STREAM
# ============================================================

N_ACC = 80

acc_phase = rng.uniform(0, 1, N_ACC)

acc_scatter = ax.scatter(
    np.zeros(N_ACC),
    np.zeros(N_ACC),
    s=rng.uniform(2, 8, N_ACC),
    c="#ffb36a",
    alpha=0,
    linewidths=0,
    zorder=6
)


# Accretion disk
disk_theta = np.linspace(
    0,
    2*np.pi,
    200
)

disk_line, = ax.plot(
    [],
    [],
    color="#7edcff",
    lw=1.2,
    alpha=0,
    zorder=8
)

disk_line2, = ax.plot(
    [],
    [],
    color="#ffffff",
    lw=3.5,
    alpha=0,
    zorder=7
)


# ============================================================
# NOVA WIND PARTICLES
# ============================================================

N_WIND = 350

wind_angle = rng.uniform(
    0,
    2*np.pi,
    N_WIND
)

wind_speed = rng.uniform(
    0.65,
    1.15,
    N_WIND
)

wind_birth = rng.uniform(
    0,
    1,
    N_WIND
)

wind_sizes = rng.uniform(
    1,
    6,
    N_WIND
)

wind_colors = np.ones(
    (N_WIND, 4)
)

wind_rgb = hex_to_rgb("#65dfff")

wind_colors[:, :3] = wind_rgb
wind_colors[:, 3] = 0


wind_scatter = ax.scatter(
    np.zeros(N_WIND),
    np.zeros(N_WIND),
    s=wind_sizes,
    c=wind_colors,
    linewidths=0,
    zorder=4
)


# ============================================================
# FAST EJECTA
# ============================================================

N_FAST = 220

fast_angle = rng.uniform(
    0,
    2*np.pi,
    N_FAST
)

fast_speed = rng.uniform(
    0.9,
    1.3,
    N_FAST
)

fast_birth = rng.uniform(
    0,
    1,
    N_FAST
)

fast_sizes = rng.uniform(
    1.5,
    6,
    N_FAST
)

fast_colors = np.ones(
    (N_FAST, 4)
)

fast_colors[:, :3] = hex_to_rgb("#fff1c4")
fast_colors[:, 3] = 0


fast_scatter = ax.scatter(
    np.zeros(N_FAST),
    np.zeros(N_FAST),
    s=fast_sizes,
    c=fast_colors,
    linewidths=0,
    zorder=5
)


# ============================================================
# SHOCKED SHELL
# ============================================================

shock_shells = []

shock_settings = [
    (18, 0.010),
    (11, 0.018),
    (7,  0.035),
    (4,  0.080),
    (2,  0.55),
]


for lw, alpha in shock_settings:

    c = Circle(
        (WD_X, WD_Y),
        0.01,
        fill=False,
        edgecolor="#f7fbff",
        linewidth=lw,
        alpha=0,
        zorder=9
    )

    ax.add_patch(c)
    shock_shells.append(c)


# Hot inner shock
inner_shock = Circle(
    (WD_X, WD_Y),
    0.01,
    fill=False,
    edgecolor="#75e8ff",
    linewidth=2,
    alpha=0,
    zorder=9
)

ax.add_patch(inner_shock)


# ============================================================
# RADIAL SHOCK FILAMENTS
# ============================================================

N_FILAMENTS = 30

filament_angles = np.linspace(
    0,
    2*np.pi,
    N_FILAMENTS,
    endpoint=False
)

filament_angles += rng.uniform(
    -0.05,
    0.05,
    N_FILAMENTS
)

filament_lines = []

for angle in filament_angles:

    line, = ax.plot(
        [],
        [],
        color="#b8eeff",
        lw=rng.uniform(0.4, 1.1),
        alpha=0,
        zorder=8
    )

    filament_lines.append(
        (line, angle)
    )


# ============================================================
# FLASH
# ============================================================

flash_scatter = ax.scatter(
    [WD_X],
    [WD_Y],
    s=10,
    c="white",
    alpha=0,
    linewidths=0,
    zorder=20
)


# ============================================================
# OBJECT LABELS
# ============================================================

comp_label = ax.text(
    COMP_X,
    COMP_Y - 0.20,
    "DONOR STAR",
    color="#b78665",
    fontsize=7,
    ha="center"
)


wd_label = ax.text(
    WD_X,
    WD_Y - 0.18,
    "WHITE DWARF",
    color="#789cb6",
    fontsize=7,
    ha="center"
)


# ============================================================
# STAGE LABEL
# ============================================================

stage_text = ax.text(
    -1.52,
    0.78,
    "",
    color="white",
    fontsize=15,
    weight="bold"
)


stage_sub = ax.text(
    -1.52,
    0.70,
    "",
    color="#7992a8",
    fontsize=8
)


# ============================================================
# PHYSICS LABELS
# ============================================================

shock_label = ax.text(
    0,
    0,
    "SHOCK",
    color="#d9f7ff",
    fontsize=8,
    weight="bold",
    alpha=0
)


shell_label = ax.text(
    0,
    0,
    "OPTICALLY THICK\nSHOCKED SHELL",
    color="white",
    fontsize=8,
    weight="bold",
    alpha=0,
    ha="center"
)


# ============================================================
# BRIGHTNESS INDICATOR
# ============================================================

indicator = fig.add_axes([
    0.78,
    0.82,
    0.17,
    0.075
])

indicator.set_xlim(0, 1)
indicator.set_ylim(0, 1)
indicator.axis("off")


indicator.text(
    0,
    0.82,
    "OPTICAL BRIGHTNESS",
    color="#6f8498",
    fontsize=7
)


brightness_bg = FancyBboxPatch(
    (0, 0.18),
    1,
    0.30,
    boxstyle="round,pad=0.02",
    facecolor="#111a24",
    edgecolor="#263848",
    linewidth=0.8
)

indicator.add_patch(
    brightness_bg
)


brightness_bar = FancyBboxPatch(
    (0, 0.18),
    0.03,
    0.30,
    boxstyle="round,pad=0.02",
    facecolor="#dcefff",
    edgecolor="none"
)

indicator.add_patch(
    brightness_bar
)


# ============================================================
# TIMELINE
# ============================================================

timeline_ax = fig.add_axes([
    0.055,
    0.045,
    0.89,
    0.035
])

timeline_ax.set_xlim(0, 1)
timeline_ax.set_ylim(0, 1)
timeline_ax.axis("off")


timeline_ax.plot(
    [0, 1],
    [0.5, 0.5],
    color="#1c3141",
    lw=2
)


timeline_progress, = timeline_ax.plot(
    [0, 0],
    [0.5, 0.5],
    color="#8adfff",
    lw=2
)


timeline_dot = timeline_ax.scatter(
    [0],
    [0.5],
    s=25,
    c="white",
    zorder=5
)


# ============================================================
# PHASE BOUNDARIES
# ============================================================

T_ACCRETION_END = 0.20
T_RUNAWAY_END = 0.31
T_WIND_END = 0.48
T_SHOCK_END = 0.62
T_SUPERBRIGHT_END = 0.80


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    f = frame / (NFRAMES - 1)

    # --------------------------------------------------------
    # TIMELINE
    # --------------------------------------------------------

    timeline_progress.set_data(
        [0, f],
        [0.5, 0.5]
    )

    timeline_dot.set_offsets(
        [[f, 0.5]]
    )


    # --------------------------------------------------------
    # STAGE TEXT
    # --------------------------------------------------------

    if f < T_ACCRETION_END:

        stage = "ACCRETION"
        subtitle = "Hydrogen-rich matter accumulates on the white dwarf"

    elif f < T_RUNAWAY_END:

        stage = "THERMONUCLEAR RUNAWAY"
        subtitle = "The nova eruption begins"

    elif f < T_WIND_END:

        stage = "NOVA WIND"
        subtitle = "Ejected material expands at high velocity"

    elif f < T_SHOCK_END:

        stage = "SHOCK FORMATION"
        subtitle = "Fast ejecta interact with slower material"

    elif f < T_SUPERBRIGHT_END:

        stage = "OPTICALLY THICK SHOCKED SHELL"
        subtitle = "A large luminous shell dominates the optical output"

    else:

        stage = "SUPERBRIGHT NOVA"
        subtitle = "The shell expands and gradually becomes optically thin"


    stage_text.set_text(stage)
    stage_sub.set_text(subtitle)


    # --------------------------------------------------------
    # ACCRETION STREAM
    # --------------------------------------------------------

    if f < 0.25:

        acc_strength = smoothstep(
            f / 0.05
        )

        acc_strength *= (
            1 -
            smoothstep(
                (f - 0.20) / 0.05
            )
        )

    else:
        acc_strength = 0


    phase = (
        acc_phase
        +
        frame * 0.006
    ) % 1.0


    # Curved stream from companion to WD
    x = COMP_X * (1 - phase)

    y = (
        COMP_Y * (1 - phase)
        +
        0.18 * np.sin(
            np.pi * phase
        )
        -
        0.08 * np.sin(
            2*np.pi * phase
        )
    )


    acc_scatter.set_offsets(
        np.column_stack(
            [x, y]
        )
    )

    acc_scatter.set_alpha(
        0.65 * acc_strength
    )


    # --------------------------------------------------------
    # ACCRETION DISK
    # --------------------------------------------------------

    disk_strength = (
        smoothstep(
            f / 0.06
        )
        *
        (
            1 -
            smoothstep(
                (f - 0.27) / 0.04
            )
        )
    )


    disk_r = 0.12

    dx = (
        disk_r
        *
        np.cos(disk_theta)
    )

    dy = (
        0.035
        *
        np.sin(disk_theta)
    )


    disk_line.set_data(
        dx,
        dy + WD_Y
    )

    disk_line2.set_data(
        dx,
        dy + WD_Y
    )


    disk_line.set_alpha(
        0.75 * disk_strength
    )

    disk_line2.set_alpha(
        0.07 * disk_strength
    )


    # --------------------------------------------------------
    # THERMONUCLEAR FLASH
    # --------------------------------------------------------

    flash = np.exp(
        -((f - 0.255) / 0.022)**2
    )


    flash_scatter.set_sizes([
        80
        +
        18000 * flash
    ])

    flash_scatter.set_alpha(
        0.75 * flash
    )


    # --------------------------------------------------------
    # WD GLOW
    # --------------------------------------------------------

    p = pulse(frame, 0.11)

    eruption_glow = smoothstep(
        (f - 0.22) / 0.08
    )


    glow_base = [
        0.015,
        0.025,
        0.045,
        0.08
    ]


    for glow_artist, base in zip(
        wd_glows,
        glow_base
    ):

        glow_artist.set_alpha(
            base
            *
            (
                1
                +
                2.5 * eruption_glow
            )
            *
            (
                0.90
                +
                0.10 * p
            )
        )


    # --------------------------------------------------------
    # SLOW NOVA WIND
    # --------------------------------------------------------

    wind_start = 0.29

    wind_age = (
        f - wind_start
    )


    wind_active = smoothstep(
        wind_age / 0.06
    )


    age_each = (
        wind_age
        -
        0.12 * wind_birth
    )


    active_mask = (
        age_each > 0
    )


    radius = np.maximum(
        age_each,
        0
    )

    radius *= (
        2.0
        *
        wind_speed
    )


    wx = (
        WD_X
        +
        radius
        *
        np.cos(wind_angle)
    )

    wy = (
        WD_Y
        +
        radius
        *
        np.sin(wind_angle)
    )


    wind_scatter.set_offsets(
        np.column_stack(
            [wx, wy]
        )
    )


    current_wind_colors = (
        wind_colors.copy()
    )


    current_wind_colors[:, 3] = (
        active_mask
        *
        wind_active
        *
        np.clip(
            0.50
            -
            radius * 0.20,
            0,
            0.50
        )
    )


    wind_scatter.set_facecolors(
        current_wind_colors
    )


    # --------------------------------------------------------
    # FAST EJECTA
    # --------------------------------------------------------

    fast_start = 0.40

    fast_age = (
        f - fast_start
    )


    age_fast = (
        fast_age
        -
        0.08 * fast_birth
    )


    fast_mask = (
        age_fast > 0
    )


    fast_radius = np.maximum(
        age_fast,
        0
    )

    fast_radius *= (
        3.0
        *
        fast_speed
    )


    fx = (
        WD_X
        +
        fast_radius
        *
        np.cos(fast_angle)
    )

    fy = (
        WD_Y
        +
        fast_radius
        *
        np.sin(fast_angle)
    )


    fast_scatter.set_offsets(
        np.column_stack(
            [fx, fy]
        )
    )


    current_fast_colors = (
        fast_colors.copy()
    )


    current_fast_colors[:, 3] = (
        fast_mask
        *
        np.clip(
            0.70
            -
            fast_radius * 0.25,
            0,
            0.70
        )
    )


    fast_scatter.set_facecolors(
        current_fast_colors
    )


    # --------------------------------------------------------
    # SHOCKED SHELL
    # --------------------------------------------------------

    shock_birth = 0.49

    shock_progress = smoothstep(
        (f - shock_birth) / 0.09
    )


    if f > shock_birth:

        shell_radius = (
            0.14
            +
            1.15
            *
            (f - shock_birth)
        )

    else:

        shell_radius = 0.01


    # Fade after maximum
    thin_factor = (
        1
        -
        0.70
        *
        smoothstep(
            (f - 0.79) / 0.18
        )
    )


    shell_base_alphas = [
        0.010,
        0.018,
        0.035,
        0.080,
        0.55
    ]


    for shell, base_alpha in zip(
        shock_shells,
        shell_base_alphas
    ):

        shell.set_radius(
            shell_radius
        )

        shell.set_alpha(
            base_alpha
            *
            shock_progress
            *
            thin_factor
            *
            (
                0.88
                +
                0.12 * p
            )
        )


    inner_shock.set_radius(
        shell_radius * 0.91
    )

    inner_shock.set_alpha(
        0.45
        *
        shock_progress
        *
        thin_factor
    )


    # --------------------------------------------------------
    # SHOCK FILAMENTS
    # --------------------------------------------------------

    for line, angle in filament_lines:

        inner_r = (
            shell_radius * 0.82
        )

        outer_r = (
            shell_radius
            *
            rng.uniform(
                0.94,
                1.02
            )
        )


        x1 = (
            WD_X
            +
            inner_r
            *
            np.cos(angle)
        )

        y1 = (
            WD_Y
            +
            inner_r
            *
            np.sin(angle)
        )


        x2 = (
            WD_X
            +
            outer_r
            *
            np.cos(angle)
        )

        y2 = (
            WD_Y
            +
            outer_r
            *
            np.sin(angle)
        )


        line.set_data(
            [x1, x2],
            [y1, y2]
        )


        line.set_alpha(
            0.30
            *
            shock_progress
            *
            thin_factor
        )


    # --------------------------------------------------------
    # SHOCK LABEL
    # --------------------------------------------------------

    shock_label_alpha = (
        smoothstep(
            (f - 0.50) / 0.05
        )
        *
        (
            1 -
            smoothstep(
                (f - 0.67) / 0.06
            )
        )
    )


    shock_label.set_position(
        (
            shell_radius * 0.73,
            WD_Y + shell_radius * 0.54
        )
    )

    shock_label.set_alpha(
        shock_label_alpha
    )


    # --------------------------------------------------------
    # OPTICALLY THICK SHELL LABEL
    # --------------------------------------------------------

    shell_label_alpha = (
        smoothstep(
            (f - 0.61) / 0.06
        )
        *
        (
            1 -
            smoothstep(
                (f - 0.85) / 0.08
            )
        )
    )


    shell_label.set_position(
        (
            0,
            WD_Y - shell_radius - 0.12
        )
    )

    shell_label.set_alpha(
        shell_label_alpha
    )


    # --------------------------------------------------------
    # SUPERBRIGHT FLASH / OPTICAL OUTPUT
    # --------------------------------------------------------

    normal_brightness = (
        0.12
        +
        0.30
        *
        smoothstep(
            (f - 0.25) / 0.15
        )
    )


    shock_brightness = (
        0.58
        *
        smoothstep(
            (f - 0.53) / 0.10
        )
        *
        (
            1
            -
            0.65
            *
            smoothstep(
                (f - 0.80) / 0.17
            )
        )
    )


    brightness = np.clip(
        normal_brightness
        +
        shock_brightness,
        0.03,
        1.0
    )


    brightness_bar.set_width(
        brightness
    )


    # Brightness changes from cool white to brilliant white
    if brightness > 0.75:

        brightness_bar.set_facecolor(
            "#ffffff"
        )

    else:

        brightness_bar.set_facecolor(
            "#8edfff"
        )


    # --------------------------------------------------------
    # LARGE SUPERBRIGHT HALO
    # --------------------------------------------------------

    superbright = (
        smoothstep(
            (f - 0.58) / 0.08
        )
        *
        (
            1
            -
            smoothstep(
                (f - 0.82) / 0.16
            )
        )
    )


    flash_scatter.set_sizes([
        max(
            10,
            7000 * superbright
        )
    ])


    flash_scatter.set_alpha(
        max(
            0,
            0.16 * superbright
        )
    )


    # --------------------------------------------------------
    # Hide donor labels after eruption becomes dominant
    # --------------------------------------------------------

    object_label_alpha = (
        1
        -
        0.75
        *
        smoothstep(
            (f - 0.45) / 0.20
        )
    )


    comp_label.set_alpha(
        object_label_alpha
    )

    wd_label.set_alpha(
        object_label_alpha
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False,
    repeat=True
)


# ============================================================
# EXPORT
# ============================================================

print(
    f"Rendering {NFRAMES} frames "
    f"({DURATION} s at {FPS} fps)"
)

print(
    f"Output: {OUTFILE}"
)


if OUTPUT.lower() == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-row-mt",
            "1",
            "-deadline",
            "good",
            "-cpu-used",
            "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-preset",
            "medium",
            "-movflags",
            "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)


plt.close(fig)


# ============================================================
# RESULT
# ============================================================

size_mb = (
    os.path.getsize(OUTFILE)
    /
    (1024 * 1024)
)


print()
print("Done.")
print(
    f"File: {OUTFILE}"
)
print(
    f"Size: {size_mb:.1f} MB"
)


display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes=(
            "controls loop autoplay muted"
        )
    )
)

# Как новая становится сверхъяркой

Классическая новая возникает в тесной двойной системе, где белый карлик перетягивает богатое водородом вещество со звезды-компаньона. Когда накопившийся на поверхности белого карлика слой достигает критических условий, начинается термоядерная вспышка. Фотосфера резко расширяется, а система выбрасывает вещество с большой скоростью, формируя мощный ветер новой. Hachisu_2026_ApJ_1009_122.pdf

Однако в некоторых очень быстрых новых обычного излучения расширяющегося газа недостаточно, чтобы объяснить наблюдаемую максимальную яркость. Авторы исследования связывают необычайно яркий оптический пик с образованием ударной оболочки: более быстрый выброс взаимодействует с ранее выброшенным веществом, и далеко за пределами фотосферы белого карлика возникает сильный ударный фронт. Если сформировавшаяся оболочка становится оптически толстой, она создаёт дополнительный мощный источник оптического излучения — новая становится сверхъяркой. Hachisu_2026_ApJ_1009_122.pdf

На примере V1674 Her модели показывают возникновение сильного ударного фронта примерно через 0,32 суток после начала вспышки. Вскоре ударная оболочка становится оптически толстой, но уже примерно через сутки снова становится прозрачной, после чего в видимом диапазоне вновь начинает доминировать обычное free–free излучение расширяющегося вещества. Hachisu_2026_ApJ_1009_122.pdf

Авторы относят к сверхъярким новым нашей Галактики три объекта: V1500 Cyg, V838 Her и V1674 Her. Новая V838 Her достигает абсолютной звёздной величины около MV,max = −10.0, и её чрезвычайно яркий пик авторы также объясняют образованием оптически толстой ударной оболочки.  

Анимация является схематической визуализацией предложенного физического механизма. Размеры, скорости и временная продолжительность отдельных стадий на экране не представлены в физическом масштабе.

# How a Nova Becomes Superbright

A classical nova occurs in a close binary system in which a white dwarf accretes hydrogen-rich material from its companion star. When the accumulated envelope on the white dwarf reaches critical conditions, a thermonuclear runaway begins. The photosphere expands dramatically and the system ejects material at high velocity, producing a powerful nova wind. Hachisu_2026_ApJ_1009_122.pdf

In some very fast novae, however, the emission from the expanding ejecta alone cannot account for the observed maximum brightness. The authors associate the unusually luminous optical peak with the formation of a shocked shell: faster ejecta interact with material expelled earlier, producing a strong shock far outside the white-dwarf photosphere. If the resulting shell becomes optically thick, it provides an additional powerful source of optical radiation, producing a superbright nova. Hachisu_2026_ApJ_1009_122.pdf

For V1674 Her, the models indicate that a strong shock develops approximately 0.32 days after the outburst. The shocked shell soon becomes optically thick, but becomes optically thin again at approximately day 1, after which free–free emission from the expanding ejecta once again dominates the V-band luminosity. Hachisu_2026_ApJ_1009_122.pdf

The authors identify three Galactic superbright novae: V1500 Cyg, V838 Her, and V1674 Her. V838 Her reaches an absolute visual magnitude of approximately MV,max = −10.0, with its exceptionally luminous optical peak likewise attributed to the formation of an optically thick shocked shell.  

The animation is a schematic visualization of the proposed physical mechanism. The sizes, velocities, and durations of the individual stages shown on screen are not represented to physical scale.

----

# Part 2. Superbright Nova: Shocked Shell and Optical Light Curve

In [ ]:
# ============================================================
# SUPERBRIGHT NOVA:
# PHYSICAL EVOLUTION + LIGHT CURVE
# ============================================================
#
# Single Jupyter cell
#
# Left:
#   schematic nova ejecta / shocked shell
#
# Right:
#   schematic V-band light curve
#   free-free emission vs. additional shocked-shell emission
#
# Inspired by V838 Her analysis in:
# Hachisu & Kato (2026), ApJ 1009:122
#
# IMPORTANT:
# The plotted light curves are schematic.
# They are NOT digitized from the paper.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle
from IPython.display import Video, display
import shutil
import os


# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "webm"          # "webm" or "mp4"

FPS = 30
DURATION = 18
DPI = 120

NFRAMES = FPS * DURATION

if OUTPUT.lower() == "webm":
    OUTFILE = "superbright_nova_lightcurve.webm"
else:
    OUTFILE = "superbright_nova_lightcurve.mp4"


if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg not found. Install ffmpeg and make sure "
        "it is available in PATH."
    )


rng = np.random.default_rng(12)


# ============================================================
# HELPERS
# ============================================================

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x*x*(3.0 - 2.0*x)


def gaussian(x, mu, sigma):
    return np.exp(
        -0.5*((x-mu)/sigma)**2
    )


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor="#02050b"
)


# LEFT — physical system
ax = fig.add_axes([
    0.035,
    0.12,
    0.43,
    0.76
])

ax.set_facecolor("#02050b")
ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.axis("off")


# RIGHT — light curve
lc = fig.add_axes([
    0.54,
    0.19,
    0.41,
    0.61
])

lc.set_facecolor("#050a11")


# ============================================================
# TITLES
# ============================================================

fig.text(
    0.055,
    0.945,
    "WHEN A NOVA BECOMES SUPERBRIGHT",
    color="white",
    fontsize=20,
    weight="bold"
)

fig.text(
    0.055,
    0.902,
    "Schematic evolution of a shocked shell and the optical light curve",
    color="#B8D4E8",
    fontsize=12
)


# ============================================================
# BACKGROUND STARS — LEFT
# ============================================================

N_BG = 150

bx = rng.uniform(-1.15, 1.15, N_BG)
by = rng.uniform(-1.0, 1.0, N_BG)

bs = rng.uniform(0.3, 2.0, N_BG)

bc = np.ones((N_BG, 4))
bc[:, 3] = rng.uniform(0.04, 0.18, N_BG)

ax.scatter(
    bx,
    by,
    s=bs,
    c=bc,
    linewidths=0,
    zorder=0
)


# ============================================================
# CENTRAL WHITE DWARF
# ============================================================

CX = 0
CY = 0

glows = []

for size, alpha in [
    (1800, 0.012),
    (1000, 0.020),
    (550, 0.035),
    (250, 0.07)
]:

    s = ax.scatter(
        [CX],
        [CY],
        s=size,
        c="white",
        alpha=alpha,
        linewidths=0,
        zorder=5
    )

    glows.append(s)


star = ax.scatter(
    [CX],
    [CY],
    s=65,
    c="white",
    edgecolors="#c8eeff",
    linewidths=0.8,
    zorder=10
)


# ============================================================
# EJECTA PARTICLES
# ============================================================

N_EJECTA = 380

angles = rng.uniform(
    0,
    2*np.pi,
    N_EJECTA
)

speeds = rng.uniform(
    0.70,
    1.25,
    N_EJECTA
)

birth = rng.uniform(
    0,
    1,
    N_EJECTA
)

sizes = rng.uniform(
    1,
    5,
    N_EJECTA
)


ejecta_colors = np.ones(
    (N_EJECTA, 4)
)

ejecta_colors[:, 0] = 0.40
ejecta_colors[:, 1] = 0.85
ejecta_colors[:, 2] = 1.00
ejecta_colors[:, 3] = 0


ejecta = ax.scatter(
    np.zeros(N_EJECTA),
    np.zeros(N_EJECTA),
    s=sizes,
    c=ejecta_colors,
    linewidths=0,
    zorder=3
)


# ============================================================
# FAST EJECTA
# ============================================================

N_FAST = 180

fast_angles = rng.uniform(
    0,
    2*np.pi,
    N_FAST
)

fast_speeds = rng.uniform(
    1.1,
    1.5,
    N_FAST
)

fast_birth = rng.uniform(
    0,
    1,
    N_FAST
)

fast_sizes = rng.uniform(
    1.5,
    5.5,
    N_FAST
)


fast_colors = np.ones(
    (N_FAST, 4)
)

fast_colors[:, 0] = 1.00
fast_colors[:, 1] = 0.90
fast_colors[:, 2] = 0.60
fast_colors[:, 3] = 0


fast = ax.scatter(
    np.zeros(N_FAST),
    np.zeros(N_FAST),
    s=fast_sizes,
    c=fast_colors,
    linewidths=0,
    zorder=4
)


# ============================================================
# SHOCK SHELL
# ============================================================

shells = []

for lw, alpha in [
    (18, 0.012),
    (11, 0.025),
    (6,  0.055),
    (2.5, 0.65)
]:

    c = Circle(
        (CX, CY),
        0.01,
        fill=False,
        edgecolor="white",
        linewidth=lw,
        alpha=0,
        zorder=8
    )

    ax.add_patch(c)

    shells.append(
        (c, alpha)
    )


inner_shell = Circle(
    (CX, CY),
    0.01,
    fill=False,
    edgecolor="#70ddff",
    linewidth=1.6,
    alpha=0,
    zorder=9
)

ax.add_patch(inner_shell)


# ============================================================
# FLASH
# ============================================================

flash = ax.scatter(
    [CX],
    [CY],
    s=100,
    c="white",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# LEFT TEXT
# ============================================================

phase_text = ax.text(
    -1.05,
    0.87,
    "",
    color="white",
    fontsize=16,
    weight="bold"
)

phase_sub = ax.text(
    -1.05,
    0.77,
    "",
    color="#B7D3E7",
    fontsize=11
)


shell_text = ax.text(
    0,
    -0.65,
    "",
    color="#E8FAFF",
    fontsize=11,
    weight="bold",
    ha="center",
    alpha=0
)


# ============================================================
# LIGHT CURVE
# ============================================================

# Schematic time axis
days = np.linspace(
    0,
    8,
    500
)


# ------------------------------------------------------------
# Schematic free-free curve
# ------------------------------------------------------------

free_mag = (
    -5.4
    - 3.0 * gaussian(days, 0.75, 0.70)
    + 0.30 * days
)


# ------------------------------------------------------------
# Additional optical brightness from shocked shell
# ------------------------------------------------------------

shock_extra = (
    2.35
    *
    gaussian(
        days,
        0.55,
        0.42
    )
)


combined_mag = (
    free_mag
    -
    shock_extra
)


# ------------------------------------------------------------
# Scale peak to approximately -10
# for visual reference to V838 Her
# ------------------------------------------------------------

combined_mag += (
    -10.0
    -
    np.min(combined_mag)
)


offset = (
    combined_mag[-1]
    -
    (
        free_mag[-1]
        -
        shock_extra[-1]
    )
)

free_mag += offset


# ============================================================
# GRAPH STYLE
# ============================================================

lc.set_xlim(0, 8)
lc.set_ylim(-10.8, -4.0)

# Astronomical magnitude:
# more negative = brighter.
lc.invert_yaxis()


lc.set_xlabel(
    "DAYS AFTER OUTBURST",
    color="#B5CDE0",
    fontsize=11,
    weight="bold",
    labelpad=12
)

lc.set_ylabel(
    "ABSOLUTE V MAGNITUDE",
    color="#B5CDE0",
    fontsize=11,
    weight="bold",
    labelpad=12
)


lc.tick_params(
    colors="#AFC7D9",
    labelsize=10
)


for spine in lc.spines.values():
    spine.set_color("#52738A")
    spine.set_linewidth(1.0)


lc.grid(
    alpha=0.18,
    linewidth=0.7
)


# ============================================================
# SUPERBRIGHT REGION
# ============================================================

lc.axhspan(
    -10.8,
    -10.0,
    alpha=0.09,
    color="white"
)

lc.text(
    7.7,
    -10.15,
    "SUPERBRIGHT",
    color="#C2D8E7",
    fontsize=10,
    weight="bold",
    ha="right",
    va="center"
)


# ============================================================
# FULL REFERENCE CURVES
# ============================================================

free_reference, = lc.plot(
    days,
    free_mag,
    color="#65BFEA",
    lw=1.5,
    alpha=0.52,
    linestyle="--"
)


combined_reference, = lc.plot(
    days,
    combined_mag,
    color="#E8F7FF",
    lw=1.0,
    alpha=0.22
)


# ============================================================
# ANIMATED CURVES
# ============================================================

free_progress, = lc.plot(
    [],
    [],
    color="#69D4FF",
    lw=2.7,
    label="FREE–FREE EMISSION"
)


shock_progress, = lc.plot(
    [],
    [],
    color="white",
    lw=3.0,
    label="WITH SHOCKED SHELL"
)


point = lc.scatter(
    [],
    [],
    s=60,
    c="white",
    edgecolors="#8BE5FF",
    linewidths=1.3,
    zorder=10
)


# ============================================================
# GRAPH LABELS
# ============================================================

free_label = lc.text(
    4.6,
    -6.4,
    "FREE–FREE EMISSION",
    color="#75D9FF",
    fontsize=10,
    weight="bold",
    alpha=0
)


shock_label = lc.text(
    1.25,
    -9.65,
    "SHOCKED SHELL",
    color="white",
    fontsize=11,
    weight="bold",
    alpha=0
)


peak_label = lc.text(
    0,
    0,
    "",
    color="white",
    fontsize=11,
    weight="bold",
    alpha=0
)


# ============================================================
# V838 HER LABEL
# ============================================================

lc.text(
    0.03,
    1.055,
    "SCHEMATIC V-BAND LIGHT CURVE — V838 HER",
    transform=lc.transAxes,
    color="#C2D9E9",
    fontsize=11,
    weight="bold"
)


# ============================================================
# TIMELINE
# ============================================================

timeline = fig.add_axes([
    0.055,
    0.045,
    0.89,
    0.035
])

timeline.set_xlim(0, 1)
timeline.set_ylim(0, 1)
timeline.axis("off")


timeline.plot(
    [0, 1],
    [0.5, 0.5],
    color="#385B70",
    lw=2.5
)


timeline_progress, = timeline.plot(
    [0, 0],
    [0.5, 0.5],
    color="#83E0FF",
    lw=3
)


timeline_dot = timeline.scatter(
    [0],
    [0.5],
    s=35,
    c="white"
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    f = frame / (NFRAMES - 1)


    # --------------------------------------------------------
    # TIMELINE
    # --------------------------------------------------------

    timeline_progress.set_data(
        [0, f],
        [0.5, 0.5]
    )

    timeline_dot.set_offsets(
        [[f, 0.5]]
    )


    # --------------------------------------------------------
    # MAP ANIMATION TIME TO SCHEMATIC NOVA DAYS
    # --------------------------------------------------------

    if f < 0.65:

        current_day = (
            2.0
            *
            smoothstep(
                f / 0.65
            )
        )

    else:

        current_day = (
            2.0
            +
            6.0
            *
            smoothstep(
                (f - 0.65) / 0.35
            )
        )


    idx = np.searchsorted(
        days,
        current_day
    )

    idx = np.clip(
        idx,
        1,
        len(days)-1
    )


    # --------------------------------------------------------
    # PHASES
    # --------------------------------------------------------

    if current_day < 0.22:

        phase = "NOVA ERUPTION"
        subtitle = "Rapid expansion begins"

    elif current_day < 0.48:

        phase = "NOVA WIND"
        subtitle = "Ejecta expand away from the white dwarf"

    elif current_day < 0.85:

        phase = "SHOCK FORMATION"
        subtitle = "Fast ejecta interact with slower material"

    elif current_day < 1.5:

        phase = "OPTICALLY THICK SHELL"
        subtitle = "Shock-powered optical emission becomes important"

    else:

        phase = "DECLINE"
        subtitle = "The expanding shell becomes increasingly transparent"


    phase_text.set_text(
        phase
    )

    phase_sub.set_text(
        subtitle
    )


    # --------------------------------------------------------
    # EJECTA
    # --------------------------------------------------------

    age = np.maximum(
        current_day / 8.0
        -
        0.07 * birth,
        0
    )


    active = age > 0


    r = (
        2.0
        *
        age
        *
        speeds
    )


    ex = (
        CX
        +
        r*np.cos(angles)
    )

    ey = (
        CY
        +
        r*np.sin(angles)
    )


    ejecta.set_offsets(
        np.column_stack(
            [ex, ey]
        )
    )


    ec = ejecta_colors.copy()

    ec[:, 3] = (
        active
        *
        np.clip(
            0.55
            -
            0.25*r,
            0,
            0.55
        )
    )

    ejecta.set_facecolors(ec)


    # --------------------------------------------------------
    # FAST EJECTA
    # --------------------------------------------------------

    fast_age = np.maximum(
        (current_day - 0.30) / 8.0
        -
        0.04 * fast_birth,
        0
    )


    fast_active = (
        fast_age > 0
    )


    fr = (
        2.8
        *
        fast_age
        *
        fast_speeds
    )


    fx = (
        CX
        +
        fr*np.cos(fast_angles)
    )

    fy = (
        CY
        +
        fr*np.sin(fast_angles)
    )


    fast.set_offsets(
        np.column_stack(
            [fx, fy]
        )
    )


    fc = fast_colors.copy()

    fc[:, 3] = (
        fast_active
        *
        np.clip(
            0.65
            -
            0.30*fr,
            0,
            0.65
        )
    )

    fast.set_facecolors(fc)


    # --------------------------------------------------------
    # SHOCK SHELL
    # --------------------------------------------------------

    shell_on = smoothstep(
        (current_day - 0.35)
        / 0.30
    )


    shell_thin = (
        1.0
        -
        0.78
        *
        smoothstep(
            (current_day - 1.25)
            / 2.8
        )
    )


    shell_radius = (
        0.16
        +
        0.12 * current_day
    )


    for shell, base_alpha in shells:

        shell.set_radius(
            shell_radius
        )

        shell.set_alpha(
            base_alpha
            *
            shell_on
            *
            shell_thin
        )


    inner_shell.set_radius(
        shell_radius * 0.91
    )

    inner_shell.set_alpha(
        0.45
        *
        shell_on
        *
        shell_thin
    )


    # --------------------------------------------------------
    # CENTRAL GLOW
    # --------------------------------------------------------

    optical_boost = (
        gaussian(
            current_day,
            0.60,
            0.45
        )
    )


    for n, g in enumerate(glows):

        base = [
            0.012,
            0.020,
            0.035,
            0.07
        ][n]

        g.set_alpha(
            base
            *
            (
                1
                +
                4.0*optical_boost
            )
        )


    flash_strength = (
        gaussian(
            current_day,
            0.55,
            0.30
        )
    )


    flash.set_sizes([
        100
        +
        7000
        *
        flash_strength
    ])

    flash.set_alpha(
        0.13
        *
        flash_strength
    )


    # --------------------------------------------------------
    # SHELL TEXT
    # --------------------------------------------------------

    shell_alpha = (
        smoothstep(
            (current_day - 0.45)
            / 0.25
        )
        *
        (
            1
            -
            smoothstep(
                (current_day - 2.0)
                / 1.5
            )
        )
    )


    shell_text.set_text(
        "OPTICALLY THICK\nSHOCKED SHELL"
    )

    shell_text.set_position(
        (
            0,
            -shell_radius - 0.15
        )
    )

    shell_text.set_alpha(
        shell_alpha
    )


    # --------------------------------------------------------
    # GRAPH — FREE-FREE
    # --------------------------------------------------------

    free_progress.set_data(
        days[:idx],
        free_mag[:idx]
    )


    # --------------------------------------------------------
    # GRAPH — SHOCKED-SHELL CURVE
    # --------------------------------------------------------

    shock_progress.set_data(
        days[:idx],
        combined_mag[:idx]
    )


    point.set_offsets(
        [[
            days[idx],
            combined_mag[idx]
        ]]
    )


    # --------------------------------------------------------
    # GRAPH LABELS
    # --------------------------------------------------------

    free_label.set_alpha(
        smoothstep(
            (current_day - 0.6)
            / 1.5
        )
    )


    shock_label.set_alpha(
        smoothstep(
            (current_day - 0.45)
            / 0.25
        )
        *
        (
            1
            -
            smoothstep(
                (current_day - 2.0)
                / 1.5
            )
        )
    )


    # --------------------------------------------------------
    # PEAK LABEL
    # --------------------------------------------------------

    peak_idx = np.argmin(
        combined_mag
    )

    peak_day = days[
        peak_idx
    ]

    peak_mag = combined_mag[
        peak_idx
    ]


    peak_alpha = smoothstep(
        (current_day - peak_day)
        / 0.30
    )


    peak_label.set_position(
        (
            peak_day + 0.25,
            peak_mag + 0.28
        )
    )

    peak_label.set_text(
        r"$M_{V,\mathrm{max}}\approx-10$"
    )

    peak_label.set_alpha(
        peak_alpha
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False,
    repeat=True
)


# ============================================================
# EXPORT
# ============================================================

print(
    f"Rendering {NFRAMES} frames "
    f"({DURATION} s at {FPS} fps)"
)

print(
    f"Output: {OUTFILE}"
)


if OUTPUT.lower() == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-row-mt",
            "1",
            "-deadline",
            "good",
            "-cpu-used",
            "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-preset",
            "medium",
            "-movflags",
            "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)


plt.close(fig)


# ============================================================
# RESULT
# ============================================================

size_mb = (
    os.path.getsize(OUTFILE)
    /
    (1024*1024)
)


print()
print("Done.")
print(
    f"File: {OUTFILE}"
)
print(
    f"Size: {size_mb:.1f} MB"
)


display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes=(
            "controls loop autoplay muted"
        )
    )
)

# Сверхъяркая новая: ударная оболочка и кривая блеска

Яркость классической новой обычно можно описать излучением расширяющегося вещества, выброшенного белым карликом во время вспышки. В работе Hachisu и Kato такие кривые блеска рассчитываются с помощью моделей free–free излучения. Однако у некоторых чрезвычайно быстрых новых наблюдаемый оптический максимум оказывается значительно ярче, чем предсказывает такая модель.

Авторы связывают дополнительную яркость с образованием ударной оболочки. Более быстрый выброс взаимодействует с ранее выброшенным веществом, формируя ударный фронт далеко за пределами фотосферы белого карлика. Когда эта оболочка становится оптически толстой, её излучение создаёт дополнительный оптический пик, поднимающий кривую блеска выше уровня, ожидаемого только от free–free излучения. Hachisu_2026_ApJ_1009_122.pdf

Именно такую картину авторы находят для V838 Her. Полученная абсолютная звёздная величина в максимуме составляет около MV,max = −10.0. Ранние наблюдения указывают на вклад оптически толстой ударной оболочки, а положение V838 Her выходит за область, которую теоретическая free–free модель предсказывает для обычных новых. Поэтому в этой работе V838 Her классифицируется как сверхъяркая новая. Hachisu_2026_ApJ_1009_122.pdf

Анимация одновременно показывает физическую картину расширяющегося выброса и её проявление на кривой блеска. Голубая линия представляет ожидаемое поведение free–free излучения, белая — дополнительную яркость при наличии ударной оболочки.

Кривые в анимации являются схематическими и не представляют собой оцифрованные наблюдательные данные или Figure 5 из статьи.

# Superbright Nova: Shocked Shell and Optical Light Curve

The brightness evolution of a classical nova can normally be described by emission from the expanding material ejected by the white dwarf during the eruption. In the analysis by Hachisu and Kato, such light curves are modeled using free–free emission. In some extremely fast novae, however, the observed optical maximum is substantially brighter than the free–free model predicts.

The authors associate this additional luminosity with the formation of a shocked shell. Faster ejecta interact with material expelled earlier, producing a shock far outside the white-dwarf photosphere. When the resulting shell becomes optically thick, its radiation provides an additional source of optical luminosity, pushing the nova above the brightness expected from free–free emission alone. Hachisu_2026_ApJ_1009_122.pdf

This is the interpretation proposed for V838 Her. The study derives a peak absolute magnitude of approximately MV,max = −10.0. Its earliest observations indicate a contribution from an optically thick shocked shell, while V838 Her lies outside the theoretical free–free MMRD region expected for ordinary novae. The authors therefore identify V838 Her as a superbright nova. Hachisu_2026_ApJ_1009_122.pdf

The animation shows the physical evolution of the expanding ejecta together with its effect on the optical light curve. The blue curve represents the expected free–free emission, while the white curve illustrates the additional luminosity produced when a shocked shell contributes to the optical output.

The light curves shown in the animation are schematic and are not digitized observational data or a reproduction of Figure 5 from the paper.

---

# Part 3. Normal Nova vs. Superbright Nova

In [ ]:
# ============================================================
# NORMAL NOVA vs SUPERBRIGHT NOVA
# ============================================================
#
# Single Jupyter cell
#
# Left:
#   NORMAL NOVA
#   expanding ejecta / free-free dominated optical emission
#
# Right:
#   SUPERBRIGHT NOVA
#   fast ejecta -> shock -> optically thick shocked shell
#
# Final:
#   Galactic superbright novae identified in the study:
#   V1500 Cyg / V838 Her / V1674 Her
#
# Inspired by:
# Hachisu & Kato (2026), ApJ 1009:122
#
# IMPORTANT:
# Geometry, ejecta and brightness evolution are schematic.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle
from IPython.display import Video, display
import shutil
import os


# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "webm"          # "webm" or "mp4"

FPS = 30
DURATION = 18
DPI = 120

NFRAMES = FPS * DURATION

if OUTPUT.lower() == "webm":
    OUTFILE = "normal_vs_superbright_nova.webm"
else:
    OUTFILE = "normal_vs_superbright_nova.mp4"


if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg not found. Install ffmpeg and make sure "
        "it is available in PATH."
    )


rng = np.random.default_rng(28)


# ============================================================
# HELPERS
# ============================================================

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x*x*(3.0 - 2.0*x)


def gaussian(x, mu, sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)


def fade_window(x, start, end, edge=0.05):

    fade_in = smoothstep(
        (x-start)/edge
    )

    fade_out = (
        1.0
        -
        smoothstep(
            (x-end)/edge
        )
    )

    return fade_in * fade_out


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor="#02050b"
)


# LEFT
axL = fig.add_axes([
    0.035,
    0.13,
    0.445,
    0.72
])


# RIGHT
axR = fig.add_axes([
    0.52,
    0.13,
    0.445,
    0.72
])


for a in (axL, axR):

    a.set_facecolor("#02050b")

    a.set_xlim(
        -1.05,
        1.05
    )

    a.set_ylim(
        -0.88,
        0.88
    )

    a.set_aspect("equal")
    a.axis("off")


# ============================================================
# TITLE
# ============================================================

fig.text(
    0.055,
    0.945,
    "NORMAL NOVA vs SUPERBRIGHT NOVA",
    color="white",
    fontsize=20,
    weight="bold"
)


fig.text(
    0.055,
    0.902,
    "How an optically thick shocked shell can enhance the optical maximum",
    color="#B8D4E8",
    fontsize=12
)


# ============================================================
# DIVIDER
# ============================================================

divider = fig.add_axes([
    0.4985,
    0.14,
    0.003,
    0.70
])

divider.set_xlim(0, 1)
divider.set_ylim(0, 1)
divider.axis("off")

divider.plot(
    [0.5, 0.5],
    [0, 1],
    color="#35556B",
    lw=1.2,
    alpha=0.85
)


# ============================================================
# BACKGROUND STARS
# ============================================================

def make_background(axis, seed):

    local_rng = np.random.default_rng(seed)

    n = 120

    x = local_rng.uniform(
        -1.05,
        1.05,
        n
    )

    y = local_rng.uniform(
        -0.88,
        0.88,
        n
    )

    s = local_rng.uniform(
        0.3,
        2.0,
        n
    )

    c = np.ones((n, 4))

    c[:, 3] = local_rng.uniform(
        0.04,
        0.17,
        n
    )

    axis.scatter(
        x,
        y,
        s=s,
        c=c,
        linewidths=0,
        zorder=0
    )


make_background(
    axL,
    10
)

make_background(
    axR,
    11
)


# ============================================================
# PANEL HEADINGS
# ============================================================

normal_title = axL.text(
    0,
    0.76,
    "NORMAL NOVA",
    color="#8DDBFF",
    fontsize=16,
    weight="bold",
    ha="center"
)


normal_sub = axL.text(
    0,
    0.66,
    "FREE–FREE EMISSION",
    color="#B5D9EA",
    fontsize=11,
    weight="bold",
    ha="center"
)


super_title = axR.text(
    0,
    0.76,
    "SUPERBRIGHT NOVA",
    color="white",
    fontsize=16,
    weight="bold",
    ha="center"
)


super_sub = axR.text(
    0,
    0.66,
    "FREE–FREE + SHOCKED SHELL",
    color="#E2F7FF",
    fontsize=11,
    weight="bold",
    ha="center"
)


# ============================================================
# CENTRAL WHITE DWARFS
# ============================================================

CX = 0
CY = 0.02


def make_star(axis):

    glows = []

    for size, alpha in [
        (1700, 0.012),
        (950, 0.022),
        (500, 0.040),
        (230, 0.075)
    ]:

        g = axis.scatter(
            [CX],
            [CY],
            s=size,
            c="white",
            alpha=alpha,
            linewidths=0,
            zorder=6
        )

        glows.append(g)


    star = axis.scatter(
        [CX],
        [CY],
        s=60,
        c="white",
        edgecolors="#C9F0FF",
        linewidths=0.9,
        zorder=10
    )


    return glows, star


normal_glows, normal_star = make_star(
    axL
)

super_glows, super_star = make_star(
    axR
)


# ============================================================
# NORMAL NOVA EJECTA
# ============================================================

N_NORMAL = 350

n_angle = rng.uniform(
    0,
    2*np.pi,
    N_NORMAL
)

n_speed = rng.uniform(
    0.65,
    1.15,
    N_NORMAL
)

n_birth = rng.uniform(
    0,
    1,
    N_NORMAL
)

n_sizes = rng.uniform(
    1.0,
    5.0,
    N_NORMAL
)


n_colors = np.ones(
    (N_NORMAL, 4)
)

n_colors[:, 0] = 0.35
n_colors[:, 1] = 0.82
n_colors[:, 2] = 1.00
n_colors[:, 3] = 0


normal_ejecta = axL.scatter(
    np.zeros(N_NORMAL),
    np.zeros(N_NORMAL),
    s=n_sizes,
    c=n_colors,
    linewidths=0,
    zorder=4
)


# ============================================================
# SUPERBRIGHT — SLOW EJECTA
# ============================================================

N_SLOW = 300

s_angle = rng.uniform(
    0,
    2*np.pi,
    N_SLOW
)

s_speed = rng.uniform(
    0.55,
    0.95,
    N_SLOW
)

s_birth = rng.uniform(
    0,
    1,
    N_SLOW
)

s_sizes = rng.uniform(
    1.0,
    5.0,
    N_SLOW
)


s_colors = np.ones(
    (N_SLOW, 4)
)

s_colors[:, 0] = 0.32
s_colors[:, 1] = 0.78
s_colors[:, 2] = 1.00
s_colors[:, 3] = 0


slow_ejecta = axR.scatter(
    np.zeros(N_SLOW),
    np.zeros(N_SLOW),
    s=s_sizes,
    c=s_colors,
    linewidths=0,
    zorder=3
)


# ============================================================
# SUPERBRIGHT — FAST EJECTA
# ============================================================

N_FAST = 220

f_angle = rng.uniform(
    0,
    2*np.pi,
    N_FAST
)

f_speed = rng.uniform(
    1.05,
    1.45,
    N_FAST
)

f_birth = rng.uniform(
    0,
    1,
    N_FAST
)

f_sizes = rng.uniform(
    1.5,
    5.5,
    N_FAST
)


f_colors = np.ones(
    (N_FAST, 4)
)

f_colors[:, 0] = 1.00
f_colors[:, 1] = 0.88
f_colors[:, 2] = 0.52
f_colors[:, 3] = 0


fast_ejecta = axR.scatter(
    np.zeros(N_FAST),
    np.zeros(N_FAST),
    s=f_sizes,
    c=f_colors,
    linewidths=0,
    zorder=5
)


# ============================================================
# NORMAL NOVA EXPANDING SHELL
# ============================================================

normal_shell = Circle(
    (CX, CY),
    0.01,
    fill=False,
    edgecolor="#69D4FF",
    linewidth=1.5,
    alpha=0,
    zorder=5
)

axL.add_patch(
    normal_shell
)


# ============================================================
# SUPERBRIGHT SHOCKED SHELL
# ============================================================

shock_shells = []

for width, alpha in [
    (22, 0.010),
    (14, 0.020),
    (8,  0.045),
    (4,  0.10),
    (2,  0.70)
]:

    shell = Circle(
        (CX, CY),
        0.01,
        fill=False,
        edgecolor="white",
        linewidth=width,
        alpha=0,
        zorder=9
    )

    axR.add_patch(
        shell
    )

    shock_shells.append(
        (shell, alpha)
    )


shock_inner = Circle(
    (CX, CY),
    0.01,
    fill=False,
    edgecolor="#73DFFF",
    linewidth=1.7,
    alpha=0,
    zorder=10
)

axR.add_patch(
    shock_inner
)


# ============================================================
# FLASHES
# ============================================================

normal_flash = axL.scatter(
    [CX],
    [CY],
    s=100,
    c="#C9F4FF",
    alpha=0,
    linewidths=0,
    zorder=15
)


super_flash = axR.scatter(
    [CX],
    [CY],
    s=100,
    c="white",
    alpha=0,
    linewidths=0,
    zorder=15
)


# ============================================================
# PHYSICS LABELS
# ============================================================

normal_label = axL.text(
    0,
    -0.69,
    "EXPANDING EJECTA",
    color="#9CDFFF",
    fontsize=11,
    weight="bold",
    ha="center",
    alpha=0
)


shock_label = axR.text(
    0,
    -0.69,
    "OPTICALLY THICK\nSHOCKED SHELL",
    color="white",
    fontsize=11,
    weight="bold",
    ha="center",
    alpha=0
)


fast_label = axR.text(
    0.46,
    0.23,
    "FAST EJECTA",
    color="#FFE6A7",
    fontsize=10,
    weight="bold",
    alpha=0
)


# ============================================================
# BRIGHTNESS BARS
# ============================================================

barL = fig.add_axes([
    0.11,
    0.135,
    0.29,
    0.035
])

barR = fig.add_axes([
    0.60,
    0.135,
    0.29,
    0.035
])


for b in (barL, barR):

    b.set_xlim(0, 1)
    b.set_ylim(0, 1)
    b.axis("off")

    b.plot(
        [0, 1],
        [0.5, 0.5],
        color="#35556B",
        lw=7,
        solid_capstyle="round"
    )


normal_bar, = barL.plot(
    [0, 0.05],
    [0.5, 0.5],
    color="#70D9FF",
    lw=7,
    solid_capstyle="round"
)


super_bar, = barR.plot(
    [0, 0.05],
    [0.5, 0.5],
    color="white",
    lw=7,
    solid_capstyle="round"
)


barL.text(
    0,
    1.35,
    "OPTICAL BRIGHTNESS",
    color="#B5D1E2",
    fontsize=9,
    weight="bold"
)


barR.text(
    0,
    1.35,
    "OPTICAL BRIGHTNESS",
    color="#B5D1E2",
    fontsize=9,
    weight="bold"
)


# ============================================================
# CENTRAL COMPARISON TEXT
# ============================================================

comparison = fig.text(
    0.5,
    0.84,
    "",
    color="white",
    fontsize=12,
    weight="bold",
    ha="center",
    alpha=0
)


# ============================================================
# FINAL OVERLAY
# ============================================================

final_ax = fig.add_axes([
    0,
    0,
    1,
    1
])

final_ax.set_xlim(0, 1)
final_ax.set_ylim(0, 1)
final_ax.axis("off")


final_bg = final_ax.scatter(
    [0.5],
    [0.5],
    s=250000,
    c="#02050b",
    alpha=0,
    linewidths=0,
    zorder=100
)


final_title = final_ax.text(
    0.5,
    0.69,
    "GALACTIC SUPERBRIGHT NOVAE",
    color="white",
    fontsize=21,
    weight="bold",
    ha="center",
    alpha=0,
    zorder=110
)


final_sub = final_ax.text(
    0.5,
    0.625,
    "identified in the study",
    color="#B9D4E6",
    fontsize=12,
    ha="center",
    alpha=0,
    zorder=110
)


# Three objects
final_names = []

for x, name, mag in [
    (
        0.25,
        "V1500 CYG",
        r"$M_{V,\max}\approx-10.4$"
    ),
    (
        0.50,
        "V838 HER",
        r"$M_{V,\max}\approx-10.0$"
    ),
    (
        0.75,
        "V1674 HER",
        r"$M_{V,\max}\approx-10.4$"
    )
]:

    name_text = final_ax.text(
        x,
        0.48,
        name,
        color="white",
        fontsize=17,
        weight="bold",
        ha="center",
        alpha=0,
        zorder=110
    )


    mag_text = final_ax.text(
        x,
        0.415,
        mag,
        color="#9FDCFA",
        fontsize=12,
        weight="bold",
        ha="center",
        alpha=0,
        zorder=110
    )


    dot = final_ax.scatter(
        [x],
        [0.555],
        s=90,
        c="white",
        edgecolors="#7DDFFF",
        linewidths=1.5,
        alpha=0,
        zorder=110
    )


    final_names.append(
        (
            name_text,
            mag_text,
            dot
        )
    )


final_note = final_ax.text(
    0.5,
    0.285,
    "A working hypothesis: superbrightness is produced by\n"
    "an optically thick shocked shell near optical maximum",
    color="#C1D8E7",
    fontsize=12,
    ha="center",
    linespacing=1.5,
    alpha=0,
    zorder=110
)


# ============================================================
# TIMELINE
# ============================================================

timeline = fig.add_axes([
    0.055,
    0.045,
    0.89,
    0.035
])

timeline.set_xlim(0, 1)
timeline.set_ylim(0, 1)
timeline.axis("off")


timeline.plot(
    [0, 1],
    [0.5, 0.5],
    color="#385B70",
    lw=2.5
)


timeline_progress, = timeline.plot(
    [0, 0],
    [0.5, 0.5],
    color="#83E0FF",
    lw=3
)


timeline_dot = timeline.scatter(
    [0],
    [0.5],
    s=35,
    c="white",
    zorder=10
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    t = frame / (NFRAMES - 1)


    # --------------------------------------------------------
    # TIMELINE
    # --------------------------------------------------------

    timeline_progress.set_data(
        [0, t],
        [0.5, 0.5]
    )

    timeline_dot.set_offsets(
        [[t, 0.5]]
    )


    # --------------------------------------------------------
    # PHYSICAL TIME
    #
    # Main comparison occupies first ~82% of animation.
    # --------------------------------------------------------

    physical_t = np.clip(
        t / 0.82,
        0,
        1
    )


    # --------------------------------------------------------
    # NORMAL NOVA
    # --------------------------------------------------------

    normal_age = np.maximum(
        physical_t
        -
        0.10*n_birth,
        0
    )


    normal_r = (
        0.90
        *
        normal_age
        *
        n_speed
    )


    nx = (
        CX
        +
        normal_r
        *
        np.cos(n_angle)
    )

    ny = (
        CY
        +
        normal_r
        *
        np.sin(n_angle)
    )


    normal_ejecta.set_offsets(
        np.column_stack(
            [nx, ny]
        )
    )


    nc = n_colors.copy()

    nc[:, 3] = (
        (normal_age > 0)
        *
        np.clip(
            0.60
            -
            0.35*normal_r,
            0,
            0.60
        )
    )


    normal_ejecta.set_facecolors(
        nc
    )


    normal_shell_r = (
        0.10
        +
        0.64*physical_t
    )


    normal_shell.set_radius(
        normal_shell_r
    )

    normal_shell.set_alpha(
        0.42
        *
        smoothstep(
            physical_t/0.12
        )
        *
        (
            1
            -
            0.70
            *
            smoothstep(
                (physical_t-0.60)/0.35
            )
        )
    )


    # Normal optical peak
    normal_brightness = (
        0.10
        +
        0.62
        *
        gaussian(
            physical_t,
            0.30,
            0.20
        )
    )


    normal_flash.set_sizes([
        150
        +
        3600*normal_brightness
    ])

    normal_flash.set_alpha(
        0.075
        +
        0.08*normal_brightness
    )


    normal_bar.set_data(
        [0, normal_brightness],
        [0.5, 0.5]
    )


    normal_label.set_alpha(
        smoothstep(
            (physical_t-0.12)/0.12
        )
        *
        (
            1
            -
            smoothstep(
                (physical_t-0.78)/0.18
            )
        )
    )


    # --------------------------------------------------------
    # NORMAL WD GLOW
    # --------------------------------------------------------

    for i, glow in enumerate(
        normal_glows
    ):

        base = [
            0.012,
            0.022,
            0.040,
            0.075
        ][i]

        glow.set_alpha(
            base
            *
            (
                1
                +
                2.0*normal_brightness
            )
        )


    # --------------------------------------------------------
    # SUPERBRIGHT — SLOW EJECTA
    # --------------------------------------------------------

    slow_age = np.maximum(
        physical_t
        -
        0.10*s_birth,
        0
    )


    sr = (
        0.72
        *
        slow_age
        *
        s_speed
    )


    sx = (
        CX
        +
        sr*np.cos(s_angle)
    )

    sy = (
        CY
        +
        sr*np.sin(s_angle)
    )


    slow_ejecta.set_offsets(
        np.column_stack(
            [sx, sy]
        )
    )


    sc = s_colors.copy()

    sc[:, 3] = (
        (slow_age > 0)
        *
        np.clip(
            0.52
            -
            0.30*sr,
            0,
            0.52
        )
    )


    slow_ejecta.set_facecolors(
        sc
    )


    # --------------------------------------------------------
    # SUPERBRIGHT — FAST EJECTA
    # --------------------------------------------------------

    fast_start = 0.22


    fast_age = np.maximum(
        physical_t
        -
        fast_start
        -
        0.06*f_birth,
        0
    )


    fr = (
        1.18
        *
        fast_age
        *
        f_speed
    )


    fx = (
        CX
        +
        fr*np.cos(f_angle)
    )

    fy = (
        CY
        +
        fr*np.sin(f_angle)
    )


    fast_ejecta.set_offsets(
        np.column_stack(
            [fx, fy]
        )
    )


    fc = f_colors.copy()

    fc[:, 3] = (
        (fast_age > 0)
        *
        np.clip(
            0.72
            -
            0.40*fr,
            0,
            0.72
        )
    )


    fast_ejecta.set_facecolors(
        fc
    )


    fast_label.set_alpha(
        fade_window(
            physical_t,
            0.25,
            0.55,
            0.08
        )
    )


    # --------------------------------------------------------
    # SHOCK FORMATION
    # --------------------------------------------------------

    shock_start = 0.38


    shock_on = smoothstep(
        (physical_t-shock_start)
        / 0.10
    )


    shock_fade = (
        1
        -
        0.78
        *
        smoothstep(
            (physical_t-0.72)
            / 0.25
        )
    )


    shock_radius = (
        0.20
        +
        0.72
        *
        max(
            physical_t-shock_start,
            0
        )
    )


    for shell, base_alpha in shock_shells:

        shell.set_radius(
            shock_radius
        )

        shell.set_alpha(
            base_alpha
            *
            shock_on
            *
            shock_fade
        )


    shock_inner.set_radius(
        shock_radius*0.91
    )

    shock_inner.set_alpha(
        0.48
        *
        shock_on
        *
        shock_fade
    )


    # --------------------------------------------------------
    # SUPERBRIGHT OPTICAL PEAK
    # --------------------------------------------------------

    base_brightness = (
        0.10
        +
        0.55
        *
        gaussian(
            physical_t,
            0.30,
            0.20
        )
    )


    shock_brightness = (
        0.42
        *
        gaussian(
            physical_t,
            0.50,
            0.14
        )
    )


    super_brightness = np.clip(
        base_brightness
        +
        shock_brightness,
        0,
        1
    )


    super_flash.set_sizes([
        150
        +
        9000*super_brightness
    ])


    super_flash.set_alpha(
        0.07
        +
        0.12*super_brightness
    )


    super_bar.set_data(
        [0, super_brightness],
        [0.5, 0.5]
    )


    # --------------------------------------------------------
    # SUPERBRIGHT WD GLOW
    # --------------------------------------------------------

    for i, glow in enumerate(
        super_glows
    ):

        base = [
            0.012,
            0.022,
            0.040,
            0.075
        ][i]

        glow.set_alpha(
            base
            *
            (
                1
                +
                4.0*super_brightness
            )
        )


    # --------------------------------------------------------
    # SHOCK LABEL
    # --------------------------------------------------------

    shock_label.set_position(
        (
            0,
            -shock_radius-0.13
        )
    )


    shock_label.set_alpha(
        fade_window(
            physical_t,
            0.42,
            0.78,
            0.09
        )
    )


    # --------------------------------------------------------
    # COMPARISON MESSAGE
    # --------------------------------------------------------

    comparison_alpha = (
        fade_window(
            physical_t,
            0.43,
            0.72,
            0.07
        )
    )


    comparison.set_text(
        "ADDITIONAL OPTICAL EMISSION"
    )

    comparison.set_alpha(
        comparison_alpha
    )


    # --------------------------------------------------------
    # FINAL SCREEN
    # --------------------------------------------------------

    final_progress = smoothstep(
        (t-0.74)/0.04
    )


    final_bg.set_alpha(
        0.94*final_progress
    )


    final_title.set_alpha(
        final_progress
    )

    final_sub.set_alpha(
        final_progress
    )


    for name_text, mag_text, dot in final_names:

        name_text.set_alpha(
            final_progress
        )

        mag_text.set_alpha(
            final_progress
        )

        dot.set_alpha(
            final_progress
        )


    final_note.set_alpha(
        smoothstep(
            (t-0.77)/0.04
        )
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False,
    repeat=True
)


# ============================================================
# EXPORT
# ============================================================

print(
    f"Rendering {NFRAMES} frames "
    f"({DURATION} s at {FPS} fps)"
)

print(
    f"Output: {OUTFILE}"
)


if OUTPUT.lower() == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-row-mt",
            "1",
            "-deadline",
            "good",
            "-cpu-used",
            "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt",
            "yuv420p",
            "-preset",
            "medium",
            "-movflags",
            "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)


plt.close(fig)


# ============================================================
# RESULT
# ============================================================

size_mb = (
    os.path.getsize(OUTFILE)
    /
    (1024*1024)
)


print()
print("Done.")
print(
    f"File: {OUTFILE}"
)
print(
    f"Size: {size_mb:.1f} MB"
)


display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes=(
            "controls loop autoplay muted"
        )
    )
)

# Обычная новая и сверхъяркая новая

Во время вспышки классической новой белый карлик выбрасывает быстро расширяющееся вещество. В обычном случае наблюдаемую оптическую яркость можно воспроизвести моделью free–free излучения этого выброса. Именно так в исследовании Hachisu и Kato описываются пять из шести рассмотренных кандидатов: их максимальная абсолютная звёздная величина остаётся слабее −10, а кривые блеска согласуются с моделями free–free излучения. Hachisu_2026_ApJ_1009_122.pdf

У сверхъяркой новой картина отличается. Более быстрый выброс взаимодействует с ранее выброшенным веществом и создаёт сильный ударный фронт далеко за пределами фотосферы белого карлика. Возникающая ударная оболочка становится оптически толстой и добавляет собственное мощное оптическое излучение. Поэтому максимум такой новой оказывается значительно ярче того, который можно объяснить только free–free излучением. Авторы рассматривают этот механизм как рабочую гипотезу происхождения сверхъяркости. Hachisu_2026_ApJ_1009_122.pdf

В исследовании к сверхъярким новым нашей Галактики отнесены три объекта: V1500 Cyg, V838 Her и V1674 Her. V1500 Cyg и V1674 Her достигают примерно MV,max ≈ −10.4, а для V838 Her авторы получают MV,max ≈ −10.0. Анализ V838 Her показывает, что её исключительно яркий максимум нельзя объяснить обычной free–free моделью; авторы связывают дополнительную светимость с оптически толстой ударной оболочкой.  

Анимация представляет схематическое сравнение двух физических сценариев. Размеры оболочек, распределение выброса, относительная яркость и продолжительность стадий показаны не в физическом масштабе.

# Normal Nova vs. Superbright Nova

During a classical nova eruption, a white dwarf ejects rapidly expanding material. In a normal nova, the observed optical luminosity can be reproduced by free–free emission from the ejecta. In the analysis by Hachisu and Kato, five of the six candidate novae examined behave this way: their peak absolute magnitudes remain fainter than −10, and their light curves are reproduced by free–free emission models. Hachisu_2026_ApJ_1009_122.pdf

A superbright nova behaves differently. Faster ejecta interact with material expelled earlier, producing a strong shock far outside the white-dwarf photosphere. The resulting shocked shell becomes optically thick and provides an additional powerful source of optical radiation. The nova can therefore reach a substantially brighter maximum than would be expected from free–free emission alone. The authors describe this mechanism as a working hypothesis for the origin of nova superbrightness. Hachisu_2026_ApJ_1009_122.pdf

The study identifies three Galactic superbright novae: V1500 Cyg, V838 Her, and V1674 Her. V1500 Cyg and V1674 Her reach approximately MV,max ≈ −10.4, while the authors derive MV,max ≈ −10.0 for V838 Her. Their analysis of V838 Her indicates that its exceptionally luminous maximum cannot be explained by the standard free–free model alone, and associates the additional luminosity with an optically thick shocked shell.  

The animation is a schematic comparison of the two physical scenarios. Shell sizes, ejecta distributions, relative brightness, and the durations of individual stages are not shown to physical scale.